# UTS PjBL Kecerdasan Buatan & Sistem Cerdas — TIM 12
## Bot Game AI Connect-Four Berbasis Depth-Limited Minimax & Heuristic Evaluation Function

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zysrnh/ConnectFour_UTS/blob/main/Praktikum_AI_UTS_TIM12.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repository-181717?logo=github)](https://github.com/zysrnh/ConnectFour_UTS)
[![Live Demo PWA](https://img.shields.io/badge/Live%20Demo-GitHub%20Pages-2563EB?logo=googlechrome&logoColor=white)](https://zysrnh.github.io/ConnectFour_UTS/)

---

### Identitas Kelompok TIM 12
| Peran | Nama Mahasiswa | NIM | Tanggung Jawab Utama |
| :--- | :--- | :--- | :--- |
| **Mahasiswa A** | **Vieri Athorid Yuda** | `250414001` | *AI System Architect / Modeler* (State Space, Heuristic Evaluation Function, Minimax Core) |
| **Mahasiswa B** | **Zaki Yusron Hasyimmi** | `250414010` | *Lead Developer / Tester* (Interactive Arena, Benchmark Node & Speed, Web PWA Integration) |

- **Institusi:** Program Studi S1 Teknik Informatika / Sistem Informasi — Institut Digital Ekonomi (IDE) LPKIA Bandung
- **Mata Kuliah:** Kecerdasan Buatan & Sistem Cerdas
- **Dosen Pengampu:** Wahyu Nurjaya WK, S.T., M.Kom.
- **Tahun Akademik:** 2026

---

### Daftar Isi (Table of Contents)
1. [Bagian 1: Pemodelan AI & Relevansi Materi Sesi 1–4](#bagian-1-pemodelan-ai--relevansi-dengan-materi-sesi-14)
2. [Bagian 2: Representasi Papan & Aturan Permainan Connect-Four](#bagian-2-representasi-papan--aturan-permainan-connect-four)
3. [Bagian 3: Fungsi Evaluasi Papan (Heuristic Evaluation Function)](#bagian-3-fungsi-evaluasi-papan-heuristic-evaluation-function)
4. [Bagian 4: Implementasi Depth-Limited Minimax & Alpha-Beta Pruning](#bagian-4-implementasi-depth-limited-minimax--alpha-beta-pruning-2p--4p-paranoid)
5. [Bagian 5: Pengujian Terstruktur & Benchmark Efisiensi Node](#bagian-5-pengujian-terstruktur--benchmark-efisiensi-node)
6. [Bagian 6A: Arena Teks Konsol Cadangan (play_game)](#bagian-6a-arena-teks-konsol-cadangan-play_game)
7. [Bagian 6B: Arena UI Retro Interaktif Google Colab (Python Kernel Bridge)](#bagian-6b-arena-ui-retro-3-layar-interaktif-di-google-colab-python-kernel--web-ui-identik)

---

### Cara Menjalankan di Google Colab
1. Klik **`▶ Jalankan semua`** (menu **Runtime → Run all**) pada toolbar Google Colab. Seluruh modul Python, pengujian taktis, benchmark node, dan simulasi pertandingan akan tereksekusi otomatis secara berurutan.
2. Scroll ke sel paling bawah (**Bagian 6B**) untuk memainkan **Arena Retro Connect-Four** secara interaktif langsung di dalam Colab yang terhubung langsung ke **Python Kernel**!

---
## Bagian 1: Pemodelan AI & Relevansi dengan Materi Sesi 1–4

Implementasi proyek ini secara komprehensif mengintegrasikan materi perkuliahan dari **Sesi 1 hingga Sesi 4**:

### 1. Arsitektur Sistem Cerdas & Intelligent Agent (Sesi 1)
- **Model PEAS (Performance, Environment, Actuators, Sensors):**
  - **Performance Measure ($P$):** Menang $4$-in-a-row ($+100.000$), memblokir ancaman $3$-segaris lawan ($-80$), mengontrol petak kolom tengah ($+4$), dan meminimalkan kedalaman langkah kemenangan.
  - **Environment ($E$):** Papan kisi diskrit $2$D ($6 \times 7$ s.d. $8 \times 10$), *Fully Observable* (keadaan papan terlihat utuh), *Deterministic* (tanpa faktor stokastik/acak), *Sequential turn-based zero-sum*.
  - **Actuators ($A$):** Menjatuhkan bidak ke kolom indeks valid ($0 \dots C-1$).
  - **Sensors ($S$):** Matriks grid status petak, deteksi giliran pemain aktif, dan daftar kolom yang belum penuh.
- **Komponen Sistem Cerdas:**
  - **Input:** Matriks posisi papan Connect-Four saat ini.
  - **Knowledge Base:** Aturan kemenangan $4$-in-a-row dan bobot heuristik jendela geser (`EVAL_WEIGHTS`).
  - **Inference Engine:** Algoritma *Depth-Limited Minimax* dan *Alpha-Beta Pruning*.
  - **Output:** Kolom langkah optimal $[0 \dots C-1]$, estimasi skor, dan statistik evaluasi node.

### 2. Ruang Keadaan & Search Tree (Sesi 2)
- Ruang keadaan dimodelkan sebagai *Game Tree* berakar pada posisi papan saat ini.
- **Faktor Percabangan (*Branching Factor*):** $b \le 7$ pada papan standar (maksimal $7$ kolom yang dapat dipilih).
- **Kedalaman Maksimal (*Max Depth*):** $m = 42$ (jumlah petak papan $6 \times 7$).
- **Kompleksitas Exhaustive Search:** $\mathcal{O}(b^m) \approx 7^{42}$ ($> 4,53 \times 10^{12}$ konfigurasi valid). Penelusuran tuntas mustahil dilakukan dalam batas waktu komputasi realtime ($< 1$ detik).

### 3. Heuristic Function $h(n)$ (Sesi 3)
Ketika batas kedalaman pencarian tercapai pada posisi non-terminal (`depth >= max_depth`), fungsi evaluasi $f(s)$ bertindak sebagai fungsi heuristik terarah untuk menaksir keunggulan konfigurasi papan bagi Bot AI tanpa perlu menelusuri hingga akhir permainan.

### 4. Adversarial Search: Minimax & Alpha-Beta Pruning (Sesi 4)
Permainan bersifat *zero-sum 2-player*: **Bot AI (`2` / `X`) sebagai MAX** dan **Lawan (`1` / `O`) sebagai MIN**.
Formulasi fungsi nilai Minimax dinyatakan sebagai:

$$V(s) = \begin{cases} 
\text{Eval}(s) & \text{jika } \text{Terminal}(s) \lor \text{depth} = d \\
\max_{a \in \text{Actions}(s)} V(\text{Result}(s, a)) & \text{jika giliran MAX (AI)} \\
\min_{a \in \text{Actions}(s)} V(\text{Result}(s, a)) & \text{jika giliran MIN (Lawan)}
\end{cases}$$

- **Mengapa Depth-Limited?** Pembatasan horizon kedalaman ($d=3$ s.d. $d=5$) memastikan bot merespons secara instan ($< 30$ ms).
- **Optimasi Alpha-Beta Pruning:** Memangkas cabang pohon saat $\alpha \ge \beta$. Dipercepat dengan *Center-First Move Ordering* $[3, 2, 4, 1, 5, 0, 6]$ yang memangkas kompleksitas waktu dari $\mathcal{O}(b^d)$ menjadi mendekati $\mathcal{O}(b^{d/2})$.

In [1]:
# =============================================================================
# 📌 PENJELASAN KODE: Setup Environment & Konstanta Global
# Inisialisasi ukuran papan (6x7), token pemain, bobot skor heuristik,
# dan preset tingkat kesulitan bot (Easy, Medium, Hard).
# =============================================================================
# Setup Environment & Konstanta Global (Sinkron 1:1 dengan js/ai.js)
import math
import random
import sys
import time
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

ROWS = 6
COLS = 7
WINDOW_LENGTH = 4

BOARD_SIZES = {
    "6x7": (6, 7),
    "7x8": (7, 8),
    "7x9": (7, 9),
    "8x10": (8, 10),
}

BLOCKED = -1  # Petak batu rintangan (mode Obstacle)
EMPTY = 0
HUMAN = 1   # Pemain 1 (Coral / 'O')
AI = 2      # Pemain 2 (Kuning / 'X')
P3 = 3      # Pemain 3 (Hijau / 'G')
P4 = 4      # Pemain 4 (Ungu / 'U')

WIN_SCORE = 100000


def get_column_order(cols=COLS):
    """Kembalikan urutan prioritas kolom dari tengah ke pinggir."""
    order = []
    if cols % 2 == 1:
        mid = cols // 2
        order.append(mid)
        for d in range(1, mid + 1):
            if mid - d >= 0:
                order.append(mid - d)
            if mid + d < cols:
                order.append(mid + d)
    else:
        left = cols // 2 - 1
        right = cols // 2
        for d in range(cols // 2):
            if left - d >= 0:
                order.append(left - d)
            if right + d < cols:
                order.append(right + d)
    return order


def get_next_player(piece, player_count=2):
    """Kembalikan giliran pemain berikutnya secara siklik (1..player_count)."""
    total = 4 if player_count >= 4 else 2
    return 1 if piece >= total else piece + 1


COLUMN_ORDER = get_column_order(COLS)  # Prioritas kolom tengah ke pinggir (standar 6x7)

EVAL_WEIGHTS = {
    "WIN": 100000,
    "AI_THREE": 12,
    "AI_TWO": 3,
    "OPP_THREE": -80,
    "OPP_TWO": -2,
    "CENTER_PER_DISC": 4,
}

DIFFICULTY_PRESETS = {
    "Easy": {"max_depth": 0, "use_pruning": False, "label": "Easy (Random Move)"},
    "Medium": {"max_depth": 3, "use_pruning": False, "label": "Medium (Depth-Limited Minimax, depth=3)"},
    "Hard": {"max_depth": 5, "use_pruning": True, "label": "Hard (Depth-Limited + Alpha-Beta, depth=5)"},
}

print("=== CEK ENVIRONMENT PRAKTIKUM AI TIM 12 ===")
print("Versi Python       :", sys.version.split()[0])
print("Matplotlib tersedia:", HAS_MATPLOTLIB)
print("Konfigurasi Papan  :", f"{ROWS} baris x {COLS} kolom (opsional: 7x8 Luas, 7x9 Ekstra)")

=== CEK ENVIRONMENT PRAKTIKUM AI TIM 12 ===
Versi Python       : 3.10.6
Matplotlib tersedia: False
Konfigurasi Papan  : 6 baris x 7 kolom (opsional: 7x8 Luas, 7x9 Ekstra)


---
## Bagian 2: Representasi Papan & Aturan Permainan Connect-Four
Papan direpresentasikan sebagai matriks 2D berukuran standar `6 × 7` (juga mendukung varian lebih luas `7 × 8` Luas & `7 × 9` Ekstra; `row 0` = baris paling atas, `row rows-1` = baris paling bawah). Setiap bidak yang dijatuhkan ke kolom akan jatuh ke baris kosong terbawah akibat gravitasi.

In [2]:
# =============================================================================
# 📌 PENJELASAN KODE: Representasi Papan & Fisika Gravitasi
# Mengatur matriks kisi 2D, gravitasi bidak jatuh ke baris kosong terbawah (get_next_open_row),
# dan validasi kondisi menang 4-segaris (horizontal, vertikal, diagonal).
# =============================================================================
def create_board(rows=ROWS, cols=COLS):
    """Buat papan kosong berukuran rows x cols (baris 0 = atas, baris terakhir = bawah)."""
    return [[EMPTY for _ in range(cols)] for _ in range(rows)]


def create_special_setup(board_mode="classic", rows=ROWS, cols=COLS):
    """Buat papan awal sesuai mode ('classic', 'obstacle') dan ukuran rows x cols."""
    board = create_board(rows, cols)
    obstacles = []
    center_cols = {cols // 2} if cols % 2 == 1 else {cols // 2 - 1, cols // 2}
    side_cols = [c for c in range(cols) if c not in center_cols]

    if board_mode == "obstacle":
        col_pick_count = 4 if cols >= 8 else 3
        chosen = random.sample(side_cols, col_pick_count)
        spots = [
            (rows - 1, chosen[0]),
            (rows - 1, chosen[1]),
            (rows - 2, chosen[1]),
            (rows - 1, chosen[2]),
        ]
        if cols >= 8 and len(chosen) >= 4:
            spots.append((rows - 1, chosen[3]))
        if cols >= 10 and len(chosen) >= 4:
            spots.append((rows - 2, chosen[3]))
        for r, c in spots:
            board[r][c] = BLOCKED
            obstacles.append([r, c])

    return {"board": board, "obstacles": obstacles}


def clone_board(board):
    """Salin papan 2D secara mendalam."""
    return [row[:] for row in board]


def print_board(board):
    """Tampilkan papan Connect-Four ke konsol sesuai dimensi papan."""
    rows = len(board)
    cols = len(board[0])
    symbol_map = {BLOCKED: "#", EMPTY: ".", HUMAN: "O", AI: "X", P3: "G", P4: "U"}
    print("  " + " ".join(str(c) for c in range(cols)))
    print(" +" + "--" * cols + "+")
    for r in range(rows):
        row_str = " ".join(symbol_map[board[r][c]] for c in range(cols))
        print(f"{r}| {row_str}|")
    print(" +" + "--" * cols + "+")


def is_valid_location(board, col):
    """Cek apakah kolom valid dan baris paling atas masih kosong."""
    cols = len(board[0])
    return 0 <= col < cols and board[0][col] == EMPTY


def get_valid_locations(board):
    """Kembalikan daftar kolom valid sesuai urutan tengah-ke-pinggir."""
    cols = len(board[0])
    return [col for col in get_column_order(cols) if is_valid_location(board, col)]


def get_next_open_row(board, col):
    """Cari indeks baris terbawah yang masih kosong pada kolom tertentu."""
    for r in range(len(board) - 1, -1, -1):
        if board[r][col] == EMPTY:
            return r
    return -1


def drop_piece(board, row, col, piece):
    """Letakkan bidak pada koordinat (row, col)."""
    board[row][col] = piece


def get_winner_info(board):
    """Cek pemenang (pemain 1..4) atau seri beserta daftar koordinat kemenangan."""
    rows = len(board)
    cols = len(board[0])

    def is_player(p):
        return 1 <= p <= 4

    # Cek horizontal
    for r in range(rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if is_player(p) and p == board[r][c + 1] == board[r][c + 2] == board[r][c + 3]:
                return p, [(r, c), (r, c + 1), (r, c + 2), (r, c + 3)]

    # Cek vertikal
    for c in range(cols):
        for r in range(rows - WINDOW_LENGTH + 1):
            p = board[r][c]
            if is_player(p) and p == board[r + 1][c] == board[r + 2][c] == board[r + 3][c]:
                return p, [(r, c), (r + 1, c), (r + 2, c), (r + 3, c)]

    # Cek diagonal turun ke kanan (\)
    for r in range(rows - WINDOW_LENGTH + 1):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if is_player(p) and p == board[r + 1][c + 1] == board[r + 2][c + 2] == board[r + 3][c + 3]:
                return p, [(r, c), (r + 1, c + 1), (r + 2, c + 2), (r + 3, c + 3)]

    # Cek diagonal naik ke kanan (/)
    for r in range(WINDOW_LENGTH - 1, rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            p = board[r][c]
            if is_player(p) and p == board[r - 1][c + 1] == board[r - 2][c + 2] == board[r - 3][c + 3]:
                return p, [(r, c), (r - 1, c + 1), (r - 2, c + 2), (r - 3, c + 3)]

    # Cek seri jika semua kolom teratas penuh
    if not get_valid_locations(board):
        return "DRAW", []

    return None, []


def check_winner(board):
    """Wrapper ringkas mengembalikan 1 (HUMAN), 2 (AI), 'DRAW', atau None."""
    winner, _ = get_winner_info(board)
    return winner


# Uji cepat papan & cek kemenangan
sample_board = create_board()
for col_idx in [3, 3, 2, 2, 4, 1]:
    r_open = get_next_open_row(sample_board, col_idx)
    drop_piece(sample_board, r_open, col_idx, AI if col_idx != 1 else HUMAN)

print_board(sample_board)
print("Status Game Contoh:", check_winner(sample_board))

  0 1 2 3 4 5 6
 +--------------+
0| . . . . . . .|
1| . . . . . . .|
2| . . . . . . .|
3| . . . . . . .|
4| . . X X . . .|
5| . O X X X . .|
 +--------------+
Status Game Contoh: None


---
## Bagian 3: Fungsi Evaluasi Papan (*Heuristic Evaluation Function*)

Karena kedalaman penelusuran dibatasi oleh parameter $d$, Bot AI membutuhkan **Heuristic Evaluation Function** $f(s)$ untuk menilai keunggulan konfigurasi papan pada daun pohon pencarian:

$$f(s) = \sum_{w \in W} \text{Skor}(w) + 4 \times (\text{BidakPusat}_{\text{AI}} - \text{BidakPusat}_{\text{Lawan}})$$

### Rincian Bobot Heuristik per Jendela 4-Petak (`EVAL_WEIGHTS`)
| Konfigurasi Jendela 4-Petak | Bobot Skor | Arti & Peran Strategis |
| :--- | :---: | :--- |
| **4 Bidak AI Terhubung** | `+100.000` | Terminal State (Kemenangan Pasti AI) |
| **3 Bidak AI + 1 Petak Kosong** | `+12` | Potensi Menyerang (Peluang 1 langkah menuju menang) |
| **2 Bidak AI + 2 Petak Kosong** | `+3` | Formasi Awal (Pengembangan bidak strategis) |
| **3 Bidak Lawan + 1 Petak Kosong** | **`-80`** | **ANCAMAN KRITIS LAWAN (Wajib diblokir seketika)** |
| **2 Bidak Lawan + 2 Petak Kosong** | `−2` | Pencegahan Dini formasi lawan |
| **Bidak pada Kolom Tengah** | `+4 / +2` | Kontrol Pusat Papan (terlibat dalam jendela terbanyak) |

> 💡 **Rasionalitas Desain:** Penalti ancaman lawan ($-80$) sengaja dibuat **hampir $7\times$ lebih berat** dibanding bonus menyerang ($+12$). Hal ini menjamin bot AI bersikap rasional dan disiplin mengutamakan pertahanan (memblokir kemenangan lawan) sebelum menyusun serangannya sendiri.

In [3]:
# =============================================================================
# 📌 PENJELASAN KODE: Fungsi Evaluasi Heuristik (Sliding Window)
# Otak penilaian posisi AI. Memeriksa setiap 4 petak jendela berurutan untuk
# memberi skor positif pada bidak AI dan penalti tajam (-80) untuk memblokir lawan.
# =============================================================================
def classify_window(window, ai_piece):
    """Klasifikasikan satu jendela 4-petak untuk 2-Player maupun 4-Player."""
    ai_count = 0
    empty_count = 0
    sole_opp = 0
    sole_opp_count = 0

    for cell in window:
        if cell == BLOCKED:
            return "blocked", 0
        if cell == EMPTY:
            empty_count += 1
        elif cell == ai_piece:
            ai_count += 1
        else:
            if sole_opp == 0:
                sole_opp = cell
                sole_opp_count = 1
            elif sole_opp == cell:
                sole_opp_count += 1
            else:
                return "mixed", 0

    if ai_count > 0 and sole_opp_count > 0:
        return "mixed", 0
    if ai_count == 4:
        return "ai_win", EVAL_WEIGHTS["WIN"]
    if sole_opp_count == 4:
        return "opp_win", -EVAL_WEIGHTS["WIN"]
    if ai_count == 3 and empty_count == 1:
        return "ai3", EVAL_WEIGHTS["AI_THREE"]
    if ai_count == 2 and empty_count == 2:
        return "ai2", EVAL_WEIGHTS["AI_TWO"]
    if sole_opp_count == 3 and empty_count == 1:
        return "opp3", EVAL_WEIGHTS["OPP_THREE"]
    if sole_opp_count == 2 and empty_count == 2:
        return "opp2", EVAL_WEIGHTS["OPP_TWO"]
    return "neutral", 0


def evaluate_window(window, ai_piece):
    """Hitung skor heuristik satu jendela berisi 4 petak dari sudut pandang ai_piece."""
    _, score = classify_window(window, ai_piece)
    return score


def analyze_board(board, ai_piece=AI, player_count=2):
    """Hitung total skor evaluasi papan (2P maupun 4P) beserta rincian fitur heuristiknya."""
    rows = len(board)
    cols = len(board[0])
    total_players = 4 if player_count >= 4 else 2
    center_cols = [cols // 2] if cols % 2 == 1 else [cols // 2 - 1, cols // 2]
    center_weight = EVAL_WEIGHTS["CENTER_PER_DISC"] if cols % 2 == 1 else EVAL_WEIGHTS["CENTER_PER_DISC"] // 2

    center_ai = 0
    opp_center_counts = {1: 0, 2: 0, 3: 0, 4: 0}
    for cc in center_cols:
        for r in range(rows):
            cell = board[r][cc]
            if cell == ai_piece:
                center_ai += 1
            elif 1 <= cell <= total_players:
                opp_center_counts[cell] += 1

    center_opp = max((opp_center_counts[p] for p in range(1, total_players + 1) if p != ai_piece), default=0)

    ai_three = 0
    ai_two = 0
    opp_three = 0
    opp_two = 0
    window_score = 0

    def process_window(w):
        nonlocal ai_three, ai_two, opp_three, opp_two, window_score
        kind, sc = classify_window(w, ai_piece)
        if kind == "ai3":
            ai_three += 1
        elif kind == "ai2":
            ai_two += 1
        elif kind == "opp3":
            opp_three += 1
        elif kind == "opp2":
            opp_two += 1
        window_score += sc

    # Horizontal
    for r in range(rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r][c + 1], board[r][c + 2], board[r][c + 3]])

    # Vertikal
    for c in range(cols):
        for r in range(rows - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r + 1][c], board[r + 2][c], board[r + 3][c]])

    # Diagonal turun (\)
    for r in range(rows - WINDOW_LENGTH + 1):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r + 1][c + 1], board[r + 2][c + 2], board[r + 3][c + 3]])

    # Diagonal naik (/)
    for r in range(WINDOW_LENGTH - 1, rows):
        for c in range(cols - WINDOW_LENGTH + 1):
            process_window([board[r][c], board[r - 1][c + 1], board[r - 2][c + 2], board[r - 3][c + 3]])

    center_score = (center_ai - center_opp) * center_weight
    return {
        "total_score": window_score + center_score,
        "center_ai": center_ai,
        "center_opp": center_opp,
        "center_score": center_score,
        "ai_three": ai_three,
        "ai_two": ai_two,
        "opp_three": opp_three,
        "opp_two": opp_two,
        "window_score": window_score,
    }


def evaluate_board(board, ai_piece=AI, player_count=2):
    """Fungsi Evaluasi Papan (Evaluation Function) untuk non-terminal state."""
    return analyze_board(board, ai_piece, player_count)["total_score"]


# Uji fungsi evaluasi pada papan contoh
eval_demo = analyze_board(sample_board, AI)
print("Rincian Evaluasi Papan Contoh (Sudut Pandang AI):")
for k, v in eval_demo.items():
    print(f"  - {k:14s}: {v}")

Rincian Evaluasi Papan Contoh (Sudut Pandang AI):
  - total_score   : 47
  - center_ai     : 2
  - center_opp    : 0
  - center_score  : 8
  - ai_three      : 1
  - ai_two        : 9
  - opp_three     : 0
  - opp_two       : 0
  - window_score  : 39


---
## Bagian 4: Implementasi Depth-Limited Minimax & Alpha-Beta Pruning (2P & 4P Paranoid)
Di bawah ini terdapat dua versi algoritma pencarian:
1. `minimax(...)`: **Depth-Limited Minimax Murni** (menjelajah seluruh cabang hingga batas `max_depth`).
2. `minimax_alpha_beta(...)`: **Depth-Limited Minimax + Alpha-Beta Pruning** (memangkas cabang saat `alpha >= beta`, mendukung mode **2-Player** klasik & **4-Player Paranoid Minimax**).

Keduanya menerapkan **penalti kedalaman** (`WIN_SCORE - depth` saat menang, `depth - WIN_SCORE` saat kalah) seperti pola pada Sesi 4 agar Bot AI memilih kemenangan tercepat dan menunda kekalahan selama mungkin.

In [4]:
# =============================================================================
# 📌 PENJELASAN KODE: Algoritma Minimax & Alpha-Beta Pruning
# Mesin penentu langkah optimal. Mengeksplorasi cabang pohon permainan dan
# memangkas ranting non-optimal saat alpha >= beta dengan prioritas kolom tengah.
# =============================================================================
def minimax(board, depth, max_depth, turn_or_max, ai_piece, stats, player_count=2):
    """Depth-Limited Minimax Murni (mendukung 2P & 4P Paranoid Minimax)."""
    stats["nodes"] += 1
    total_players = 4 if player_count >= 4 else 2
    if isinstance(turn_or_max, bool):
        turn_piece = ai_piece if turn_or_max else get_next_player(ai_piece, total_players)
    else:
        turn_piece = turn_or_max

    winner = check_winner(board)
    if winner == ai_piece:
        return WIN_SCORE - depth
    if winner == "DRAW":
        return 0
    if winner is not None:
        return depth - WIN_SCORE

    if depth >= max_depth:
        return evaluate_board(board, ai_piece, total_players)

    valid_cols = get_valid_locations(board)
    next_piece = get_next_player(turn_piece, total_players)

    if turn_piece == ai_piece:
        best_score = -math.inf
        for col in valid_cols:
            row = get_next_open_row(board, col)
            board[row][col] = turn_piece
            score = minimax(board, depth + 1, max_depth, next_piece, ai_piece, stats, total_players)
            board[row][col] = EMPTY
            if score > best_score:
                best_score = score
        return best_score

    best_score = math.inf
    for col in valid_cols:
        row = get_next_open_row(board, col)
        board[row][col] = turn_piece
        score = minimax(board, depth + 1, max_depth, next_piece, ai_piece, stats, total_players)
        board[row][col] = EMPTY
        if score < best_score:
            best_score = score
    return best_score


def minimax_alpha_beta(board, depth, max_depth, alpha, beta, turn_or_max, ai_piece, stats, player_count=2):
    """Depth-Limited Minimax dengan optimasi Alpha-Beta Pruning (2P & 4P Paranoid)."""
    stats["nodes"] += 1
    total_players = 4 if player_count >= 4 else 2
    if isinstance(turn_or_max, bool):
        turn_piece = ai_piece if turn_or_max else get_next_player(ai_piece, total_players)
    else:
        turn_piece = turn_or_max

    winner = check_winner(board)
    if winner == ai_piece:
        return WIN_SCORE - depth
    if winner == "DRAW":
        return 0
    if winner is not None:
        return depth - WIN_SCORE

    if depth >= max_depth:
        return evaluate_board(board, ai_piece, total_players)

    valid_cols = get_valid_locations(board)
    next_piece = get_next_player(turn_piece, total_players)

    if turn_piece == ai_piece:
        max_eval = -math.inf
        for col in valid_cols:
            row = get_next_open_row(board, col)
            board[row][col] = turn_piece
            eval_score = minimax_alpha_beta(
                board, depth + 1, max_depth, alpha, beta, next_piece, ai_piece, stats, total_players
            )
            board[row][col] = EMPTY
            if eval_score > max_eval:
                max_eval = eval_score
            if max_eval > alpha:
                alpha = max_eval
            if alpha >= beta:
                break  # Pemangkasan cabang MIN
        return max_eval

    min_eval = math.inf
    for col in valid_cols:
        row = get_next_open_row(board, col)
        board[row][col] = turn_piece
        eval_score = minimax_alpha_beta(
            board, depth + 1, max_depth, alpha, beta, next_piece, ai_piece, stats, total_players
        )
        board[row][col] = EMPTY
        if eval_score < min_eval:
            min_eval = eval_score
        if min_eval < beta:
            beta = min_eval
        if alpha >= beta:
            break  # Pemangkasan cabang MAX
    return min_eval


def find_best_move(board, max_depth=4, ai_piece=AI, use_pruning=True, player_count=2):
    """Cari kolom terbaik beserta skor evaluasi tiap kolom dan jumlah node."""
    working_board = clone_board(board)
    valid_cols = get_valid_locations(working_board)
    column_scores = {c: None for c in range(len(working_board[0]))}

    if not valid_cols:
        return {
            "best_col": -1,
            "best_score": 0,
            "column_scores": column_scores,
            "nodes": 0,
            "elapsed_ms": 0.0,
            "max_depth": max_depth,
            "use_pruning": use_pruning,
        }

    stats = {"nodes": 0}
    t0 = time.perf_counter()

    best_score = -math.inf
    best_col = valid_cols[0]
    alpha = -math.inf
    beta = math.inf
    next_turn = get_next_player(ai_piece, player_count)

    for col in valid_cols:
        row = get_next_open_row(working_board, col)
        working_board[row][col] = ai_piece

        if use_pruning:
            score = minimax_alpha_beta(
                working_board, 1, max_depth, alpha, beta, next_turn, ai_piece, stats, player_count
            )
        else:
            score = minimax(
                working_board, 1, max_depth, next_turn, ai_piece, stats, player_count
            )

        working_board[row][col] = EMPTY
        column_scores[col] = score

        if score > best_score:
            best_score = score
            best_col = col
        if use_pruning and best_score > alpha:
            alpha = best_score

    t1 = time.perf_counter()
    return {
        "best_col": best_col,
        "best_score": best_score,
        "column_scores": column_scores,
        "nodes": stats["nodes"],
        "elapsed_ms": round((t1 - t0) * 1000, 2),
        "max_depth": max_depth,
        "use_pruning": use_pruning,
    }


def choose_move(board, difficulty="Hard", ai_piece=AI, use_pruning=None, custom_depth=None, player_count=2):
    """Pilih langkah Bot AI sesuai tingkat kesulitan ('Easy', 'Medium', 'Hard')."""
    preset = DIFFICULTY_PRESETS.get(difficulty, DIFFICULTY_PRESETS["Hard"])

    if difficulty == "Easy" and custom_depth is None:
        valid_cols = get_valid_locations(board)
        column_scores = {c: None for c in range(len(board[0]))}
        if not valid_cols:
            return {
                "best_col": -1, "best_score": 0, "column_scores": column_scores,
                "nodes": 0, "elapsed_ms": 0.0, "max_depth": 0,
                "use_pruning": False, "is_random": True,
            }
        working_board = clone_board(board)
        t0 = time.perf_counter()
        for col in valid_cols:
            row = get_next_open_row(working_board, col)
            working_board[row][col] = ai_piece
            column_scores[col] = evaluate_board(working_board, ai_piece, player_count)
            working_board[row][col] = EMPTY
        chosen = random.choice(valid_cols)
        t1 = time.perf_counter()
        return {
            "best_col": chosen,
            "best_score": column_scores[chosen],
            "column_scores": column_scores,
            "nodes": len(valid_cols),
            "elapsed_ms": round((t1 - t0) * 1000, 2),
            "max_depth": 0,
            "use_pruning": False,
            "is_random": True,
        }

    default_depth = 4 if ((player_count >= 4 or len(board[0]) >= 10) and difficulty == "Hard") else preset["max_depth"]
    max_depth = custom_depth if custom_depth is not None else default_depth
    pruning = use_pruning if use_pruning is not None else preset["use_pruning"]
    res = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=pruning, player_count=player_count)
    res["is_random"] = False
    return res


def compare_algorithms(board, max_depth=4, ai_piece=AI, player_count=2):
    """Bandingkan Minimax Murni vs Alpha-Beta Pruning pada state papan yang sama."""
    pure = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=False, player_count=player_count)
    ab = find_best_move(board, max_depth=max_depth, ai_piece=ai_piece, use_pruning=True, player_count=player_count)
    reduction = round(((pure["nodes"] - ab["nodes"]) / pure["nodes"]) * 100, 1) if pure["nodes"] > 0 else 0.0
    return {
        "max_depth": max_depth,
        "pure": pure,
        "alphabeta": ab,
        "reduction_pct": reduction,
        "same_decision": pure["best_col"] == ab["best_col"],
    }

---
## Bagian 5: Pengujian Terstruktur & Benchmark Efisiensi Node
Sel-sel di bawah ini berjalan otomatis (tanpa `input()`) untuk menguji:
1. Ketepatan taktis Bot AI (menang 1 langkah, memblokir lawan, memilih kolom tengah di awal, Obstacle, papan `7×9`, & mode 4-Player Paranoid Minimax).
2. Perbandingan jumlah *node* yang dievaluasi dan waktu komputasi **Minimax Murni vs Alpha-Beta Pruning** pada berbagai kedalaman (`depth = 1..4`).
3. Simulasi pertandingan antar tingkat kesulitan (5 game per pasangan, mengikuti standar uji coba Sesi 4).

In [5]:
# =============================================================================
# 📌 PENJELASAN KODE: Pengujian Taktis Unit Test Otomatis
# Menguji 7 skenario taktis untuk memvalidasi AI: deteksi menang 1 langkah,
# memblokir ancaman lawan, memilih kolom tengah di awal, obstacle, dan mode 4-pemain.
# =============================================================================
# 1. Pengujian Taktis Unit Test
def run_tactical_tests():
    # Kasus A: Papan kosong -> harus pilih kolom tengah (3)
    b_empty = create_board()
    res_empty = find_best_move(b_empty, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_empty["best_col"] == 3, f"Harusnya kolom 3, dapat {res_empty['best_col']}"

    # Kasus B: AI punya 3 beruntun (kolom 0,1,2 di baris 5) -> wajib selesaikan menang di kolom 3
    b_win = create_board()
    for c in [0, 1, 2]:
        drop_piece(b_win, 5, c, AI)
    for c in [0, 1]:
        drop_piece(b_win, 4, c, HUMAN)
    res_win = find_best_move(b_win, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_win["best_col"] == 3 and res_win["best_score"] == WIN_SCORE - 1

    # Kasus C: Manusia punya 3 beruntun (kolom 4,5,6 di baris 5) -> Bot wajib blokir di kolom 3
    b_block = create_board()
    for c in [4, 5, 6]:
        drop_piece(b_block, 5, c, HUMAN)
    drop_piece(b_block, 5, 0, AI)
    drop_piece(b_block, 4, 0, AI)
    res_block = find_best_move(b_block, max_depth=3, ai_piece=AI, use_pruning=True)
    assert res_block["best_col"] == 3, f"Harusnya blokir kolom 3, dapat {res_block['best_col']}"

    # Kasus D: Keputusan Minimax Murni dan Alpha-Beta harus 100% identik
    comp = compare_algorithms(b_block, max_depth=4, ai_piece=AI)
    assert comp["same_decision"], "Keputusan Minimax vs Alpha-Beta berbeda!"

    # Kasus E: Mode Obstacle (BLOCKED = -1) menonaktifkan 4-in-a-row melewati batu
    b_obs = create_board()
    b_obs[5][2] = BLOCKED
    drop_piece(b_obs, 5, 0, AI)
    drop_piece(b_obs, 5, 1, AI)
    drop_piece(b_obs, 5, 3, AI)
    assert check_winner(b_obs) is None

    # Kasus F: Mode 4-Player (P1..P4) mendeteksi menang P3 & P4 serta memblokir ancaman lawan di 4P
    b_4p = create_board(7, 9)
    for c in [5, 6, 7]:
        drop_piece(b_4p, 6, c, P3)
    res_4p = find_best_move(b_4p, max_depth=3, ai_piece=P4, use_pruning=True, player_count=4)
    assert res_4p["best_col"] in (4, 8), f"P4 di mode 4-Player harus blokir ancaman P3 di kol 4/8, dapat {res_4p['best_col']}"
    comp_4p = compare_algorithms(b_4p, max_depth=3, ai_piece=P4, player_count=4)
    assert comp_4p["same_decision"], "Keputusan Minimax vs Alpha-Beta di mode 4P harus sama!"

    # Kasus G: Ukuran papan lebih luas (7x8, 7x9, & 8x10) mendukung pusat kolom & menang di pinggir kanan
    b_wide = create_board(8, 10)
    res_wide = find_best_move(b_wide, max_depth=2, ai_piece=AI, use_pruning=True)
    assert res_wide["best_col"] == 4, f"Papan 8x10 kosong harusnya kolom tengah kiri 4, dapat {res_wide['best_col']}"
    for c in [6, 7, 8, 9]:
        drop_piece(b_wide, 7, c, AI)
    assert check_winner(b_wide) == AI

    print("[PASS] Semua 7 pengujian taktis (termasuk Obstacle, Papan 8x10, & 4-Player Paranoid Minimax) berhasil 100%!")
    print(f"  - Papan kosong -> Kol {res_empty['best_col']} (skor {res_empty['best_score']})")
    print(f"  - Deteksi menang 1 langkah -> Kol {res_win['best_col']} (skor {res_win['best_score']})")
    print(f"  - Blokir ancaman lawan     -> Kol {res_block['best_col']} (skor {res_block['best_score']})")
    print(f"  - Konsistensi Minimax vs Alpha-Beta (d=4): Kol {comp['pure']['best_col']} == Kol {comp['alphabeta']['best_col']} "
          f"(hemat {comp['reduction_pct']}% node)")


run_tactical_tests()

[PASS] Semua 7 pengujian taktis (termasuk Obstacle, Papan 8x10, & 4-Player Paranoid Minimax) berhasil 100%!
  - Papan kosong -> Kol 3 (skor 9)
  - Deteksi menang 1 langkah -> Kol 3 (skor 99999)
  - Blokir ancaman lawan     -> Kol 3 (skor 16)
  - Konsistensi Minimax vs Alpha-Beta (d=4): Kol 3 == Kol 3 (hemat 91.7% node)


In [6]:
# =============================================================================
# 📌 PENJELASAN KODE: Benchmark Efisiensi Node (Minimax vs Alpha-Beta)
# Mengukur jumlah node yang dikunjungi dan waktu komputasi pada depth 1 s.d. 4.
# Membuktikan pemangkasan node >88% dengan keputusan langkah yang tetap 100% identik.
# =============================================================================
# 2. Benchmark Minimax Murni vs Alpha-Beta Pruning (Depth 1 s.d. 4)
bench_board = create_board()
sample_opening = [(3, HUMAN), (3, AI), (2, HUMAN), (4, AI), (2, HUMAN), (3, AI)]
for col_m, piece_m in sample_opening:
    drop_piece(bench_board, get_next_open_row(bench_board, col_m), col_m, piece_m)

print("Posisi Papan Benchmark (6 langkah pembuka):")
print_board(bench_board)

rows_bench = []
for d in [1, 2, 3, 4]:
    comp = compare_algorithms(bench_board, max_depth=d, ai_piece=AI)
    rows_bench.append({
        "Max Depth": d,
        "Kolom Dipilih": comp["alphabeta"]["best_col"],
        "Skor Eval": comp["alphabeta"]["best_score"],
        "Node Minimax": comp["pure"]["nodes"],
        "Node Alpha-Beta": comp["alphabeta"]["nodes"],
        "Penghematan Node (%)": comp["reduction_pct"],
        "Waktu Minimax (ms)": comp["pure"]["elapsed_ms"],
        "Waktu Alpha-Beta (ms)": comp["alphabeta"]["elapsed_ms"],
        "Keputusan Sama?": comp["same_decision"],
    })

df_bench = pd.DataFrame(rows_bench)
print("\n=== TABEL BENCHMARK EFISIENSI NODE ===")
print(df_bench.to_string(index=False))

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    depths = df_bench["Max Depth"]

    axes[0].bar(depths - 0.18, df_bench["Node Minimax"], width=0.35, label="Minimax Murni", color="#dc2626")
    axes[0].bar(depths + 0.18, df_bench["Node Alpha-Beta"], width=0.35, label="Alpha-Beta Pruning", color="#1d4ed8")
    axes[0].set_title("Perbandingan Jumlah Node Dievaluasi")
    axes[0].set_xlabel("Max Depth")
    axes[0].set_ylabel("Jumlah Node")
    axes[0].set_xticks(depths)
    axes[0].legend()
    axes[0].grid(axis="y", linestyle="--", alpha=0.4)

    axes[1].bar(depths, df_bench["Penghematan Node (%)"], width=0.45, color="#15803d")
    axes[1].set_title("Persentase Penghematan Node Alpha-Beta")
    axes[1].set_xlabel("Max Depth")
    axes[1].set_ylabel("Penghematan (%)")
    axes[1].set_ylim(0, 100)
    axes[1].set_xticks(depths)
    axes[1].grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

Posisi Papan Benchmark (6 langkah pembuka):
  0 1 2 3 4 5 6
 +--------------+
0| . . . . . . .|
1| . . . . . . .|
2| . . . . . . .|
3| . . . X . . .|
4| . . O X . . .|
5| . . O O X . .|
 +--------------+

=== TABEL BENCHMARK EFISIENSI NODE ===
 Max Depth  Kolom Dipilih  Skor Eval  Node Minimax  Node Alpha-Beta  Penghematan Node (%)  Waktu Minimax (ms)  Waktu Alpha-Beta (ms)  Keputusan Sama?
         1              2         27             7                7                   0.0                0.97                   0.75             True
         2              2        -51            56               31                  44.6                5.98                   3.55             True
         3              3         33           399               75                  81.2               55.84                   6.96             True
         4              2         16          2757              329                  88.1              315.59                  30.12             True


In [7]:
# =============================================================================
# 📌 PENJELASAN KODE: Simulasi Turnamen Otomatis (Bot vs Bot)
# Mensimulasikan pertandingan mandiri antar bot (Easy vs Medium, Easy vs Hard,
# Medium vs Hard) untuk menguji konsistensi kemenangan bot Hard (Alpha-Beta depth 5).
# =============================================================================
# 3. Simulasi Turnamen Otomatis Bot vs Bot (5 Pertandingan per Skenario)
def simulate_match(p1_cfg, p2_cfg, seed=None, opening=None):
    if seed is not None:
        random.seed(seed)
    board = create_board()
    moves = 0
    nodes_p1, nodes_p2 = [], []

    if opening:
        for idx, col in enumerate(opening):
            piece = HUMAN if idx % 2 == 0 else AI
            drop_piece(board, get_next_open_row(board, col), col, piece)
            moves += 1
        turn = HUMAN if len(opening) % 2 == 0 else AI
    else:
        turn = HUMAN  # HUMAN = Pemain 1, AI = Pemain 2

    while True:
        cfg = p1_cfg if turn == HUMAN else p2_cfg
        res = choose_move(
            board,
            difficulty=cfg["difficulty"],
            ai_piece=turn,
            use_pruning=cfg.get("use_pruning"),
            custom_depth=cfg.get("custom_depth"),
        )
        col = res["best_col"]
        drop_piece(board, get_next_open_row(board, col), col, turn)
        moves += 1
        if turn == HUMAN:
            nodes_p1.append(res["nodes"])
        else:
            nodes_p2.append(res["nodes"])

        winner = check_winner(board)
        if winner is not None:
            return {
                "winner": winner,
                "moves": moves,
                "avg_nodes_p1": round(sum(nodes_p1) / max(1, len(nodes_p1)), 1),
                "avg_nodes_p2": round(sum(nodes_p2) / max(1, len(nodes_p2)), 1),
            }
        turn = AI if turn == HUMAN else HUMAN


varied_openings = [(3, 3), (2, 3), (4, 3), (3, 2), (3, 4)]

scenarios = [
    ("Easy (Random) vs Medium (Depth 3 Minimax)", {"difficulty": "Easy"}, {"difficulty": "Medium"}, None),
    ("Easy (Random) vs Hard (Depth 5 Alpha-Beta)", {"difficulty": "Easy"}, {"difficulty": "Hard"}, None),
    ("Medium (Depth 3) vs Hard (Depth 5 Alpha-Beta)", {"difficulty": "Medium"}, {"difficulty": "Hard"}, varied_openings),
]

summary_rows = []
for name, p1_c, p2_c, openings in scenarios:
    p1_wins = p2_wins = draws = 0
    total_moves = []
    avg_n2 = []
    for idx in range(5):
        op = openings[idx] if openings else None
        out = simulate_match(p1_c, p2_c, seed=100 + idx, opening=op)
        total_moves.append(out["moves"])
        avg_n2.append(out["avg_nodes_p2"])
        if out["winner"] == HUMAN:
            p1_wins += 1
        elif out["winner"] == AI:
            p2_wins += 1
        else:
            draws += 1

    summary_rows.append({
        "Skenario (P1 vs P2)": name,
        "P1 Menang": p1_wins,
        "P2 Menang": p2_wins,
        "Seri": draws,
        "Rata-rata Langkah": round(sum(total_moves) / len(total_moves), 1),
        "Rata-rata Node P2": round(sum(avg_n2) / len(avg_n2), 1),
    })

df_sim = pd.DataFrame(summary_rows)
print("=== HASIL SIMULASI 5 GAME PER SKENARIO ===")
print(df_sim.to_string(index=False))

=== HASIL SIMULASI 5 GAME PER SKENARIO ===
                          Skenario (P1 vs P2)  P1 Menang  P2 Menang  Seri  Rata-rata Langkah  Rata-rata Node P2
    Easy (Random) vs Medium (Depth 3 Minimax)          0          5     0               11.6              381.2
   Easy (Random) vs Hard (Depth 5 Alpha-Beta)          0          5     0               11.6             1073.5
Medium (Depth 3) vs Hard (Depth 5 Alpha-Beta)          1          4     0               33.8             1047.9


---
## Bagian 6A: Arena Teks Konsol Cadangan (`play_game`)
Mode teks konsol Python murni jika ingin bermain lewat terminal/input teks. Ubah `JALANKAN_GAME_INTERAKTIF = True` bila ingin memainkannya lewat `input()`.

In [8]:
# =============================================================================
# 📌 PENJELASAN KODE: Arena Konsol Teks CLI (play_game)
# Menyediakan antarmuka bermain interaktif via terminal/teks konsol
# dengan visualisasi ASCII board (opsional jika tidak menggunakan UI grafis).
# =============================================================================
def play_game(difficulty="Hard", human_first=True, custom_depth=None, use_pruning=None, board_size="6x7"):
    """Arena interaktif Manusia ('O') vs Bot AI ('X') mode konsol."""
    rows, cols = BOARD_SIZES.get(board_size, (ROWS, COLS))
    board = create_board(rows, cols)
    current_turn = HUMAN if human_first else AI
    step = 0

    print("=== ARENA INTERAKTIF CONNECT-FOUR AI (TIM 12) ===")
    print(f"Ukuran Papan      : {rows}x{cols}")
    print(f"Tingkat Kesulitan : {difficulty}")
    print("Simbol Pemain     : Manusia = 'O' | Bot AI = 'X'")
    print(f"Masukkan nomor kolom 0 s.d. {cols - 1} (atau 'q' untuk keluar).\n")

    while True:
        print_board(board)
        print(f"Skor Evaluasi Papan Statis (sudut pandang AI): {evaluate_board(board, AI)}")

        if current_turn == HUMAN:
            valid = [c for c in range(cols) if is_valid_location(board, c)]
            while True:
                raw = input(f"[Giliran Manusia 'O'] Pilih kolom {valid} (atau 'q'): ").strip()
                if raw.lower() == "q":
                    print("Permainan dihentikan oleh pemain.")
                    return
                if raw.isdigit() and int(raw) in valid:
                    col = int(raw)
                    break
                print("Input tidak valid! Pilih kolom yang masih tersedia.")

            row = get_next_open_row(board, col)
            drop_piece(board, row, col, HUMAN)
            step += 1
            print(f"-> Manusia menjatuhkan bidak di Kolom {col}, Baris {row}.\n")
        else:
            print("[Giliran Bot AI 'X'] Menghitung langkah terbaik...")
            res = choose_move(
                board,
                difficulty=difficulty,
                ai_piece=AI,
                use_pruning=use_pruning,
                custom_depth=custom_depth,
            )
            col = res["best_col"]
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, AI)
            step += 1

            scores_str = " | ".join(
                f"K{c}:{res['column_scores'][c]}" if res["column_scores"][c] is not None else f"K{c}:Penuh"
                for c in range(cols)
            )
            print(f"-> Bot memilih Kolom {col} (Baris {row}) | Skor: {res['best_score']} "
                  f"| Node: {res['nodes']} | Waktu: {res['elapsed_ms']} ms")
            print(f"   Skor Evaluasi Kolom: [{scores_str}]\n")

        winner, win_cells = get_winner_info(board)
        if winner is not None:
            print_board(board)
            if winner == "DRAW":
                print(f"=== HASIL AKHIR: SERI setelah {step} langkah! ===")
            elif winner == HUMAN:
                print(f"=== SELAMAT! Manusia ('O') MENANG dalam {step} langkah! Koordinat: {win_cells} ===")
            else:
                print(f"=== BOT AI ('X') MENANG dalam {step} langkah! Koordinat: {win_cells} ===")
            break

        current_turn = AI if current_turn == HUMAN else HUMAN


# Ubah menjadi True jika ingin bermain lewat konsol teks
JALANKAN_GAME_INTERAKTIF = False

if JALANKAN_GAME_INTERAKTIF:
    play_game(difficulty="Hard", human_first=True)
else:
    print("Mode konsol standby (aman untuk Run All). Scroll ke sel paling bawah (Bagian 6B) untuk UI Retro Interaktif!")

Mode konsol standby (aman untuk Run All). Scroll ke sel paling bawah (Bagian 6B) untuk UI Retro Interaktif!


---
## Bagian 6B: Arena UI Retro 3-Layar Interaktif di Google Colab (`Python Kernel` + Web UI Identik)
Sel terakhir ini merender **UI Retro Connect-Four 3-Layar** (*Main Menu → Arena Bersih Beranimasi → Tabel Analisis Pasca-Main*) yang **sama persis dengan Demo GitHub Pages**, sekaligus menghubungkan setiap langkah Bot AI & Benchmark langsung ke fungsi **Python (`choose_move` & `compare_algorithms`)** di kernel Google Colab!

> **Penting:** Pastikan semua sel di atas sudah dijalankan (**`Runtime → Run all` / `▶ Jalankan semua`**) sebelum memainkan sel ini.

In [9]:
# =============================================================================
# 📌 PENJELASAN KODE: Bridge UI Web Retro Interaktif Google Colab
# Merender grafis Web Retro 3-layar langsung di dalam notebook Colab dan
# menghubungkan setiap klik langkah pemain ke kernel Python secara realtime.
# Modul connect4_colab_ui.py memuat template UI secara dinamis (bebas dari raw HTML).
# =============================================================================

import sys

try:
    from connect4_colab_ui import launch_retro_colab_ui
except ImportError:
    # Jika dijalankan mandiri di cloud Google Colab tanpa clone repo, unduh helper otomatis
    import urllib.request
    try:
        _url = "https://raw.githubusercontent.com/zysrnh/ConnectFour_UTS/main/connect4_colab_ui.py"
        urllib.request.urlretrieve(_url, "connect4_colab_ui.py")
        from connect4_colab_ui import launch_retro_colab_ui
    except Exception as _e:
        launch_retro_colab_ui = None

if launch_retro_colab_ui:
    launch_retro_colab_ui(choose_move, compare_algorithms)
else:
    print("Mode konsol standby. Gunakan play_game() pada Bagian 6A untuk bermain via terminal.")

[Info] IPython/Colab display hook standby. Di Google Colab, sel ini akan langsung merender UI Retro 3-Layar interaktif.
